In [ ]:
!git clone https://github.com/spMohanty/PlantVillage-Dataset

Cloning into 'PlantVillage-Dataset'...
remote: Enumerating objects: 163264, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (26/26), done.


In [ ]:
!git clone https://github.com/aldrin233/RiceDiseases-DataSet.git

In [ ]:
!pip install tensorflow_model_optimization

Multi-seed running (6 classes)


In [ ]:
# ===============================================================
# STABLE REVIEWER VERSION
# SimCLR + MADA-Lite + SE-ResNet
# Multi-seed + Frozen Encoder Evaluation + Safe TFLite
# ===============================================================

import os, random, time
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

# ================= STABLE FLOAT32 =================
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("float32")
print("Using float32 for stability.")

# ================= CONFIG =================
SOURCE_DIR = '/content/PlantVillage-Dataset/raw/segmented'
TARGET_DIR = '/content/RiceDiseases-DataSet'

IMG_SIZE = 48
BATCH_SIZE = 32

EPOCHS_DA = 20
STEPS_PER_EPOCH = 200
EPOCHS_LINEAR = 20

TEMPERATURE = 0.1
FEATURE_DIM = 64
PROJECTION_DIM = 64
NUM_DOMAINS = 2
DOMAIN_LOSS_WEIGHT = 0.10

SEEDS = [42, 123, 999]

SOURCE_CLASSES = [
   "Tomato___Bacterial_spot",
    "Corn_(maize)___Northern_Leaf_Blight",
    "Potato___Early_blight",
    "Corn_(maize)___Cercospora_leaf_spot_Gray_leaf_spot",
    "Tomato___Septoria_leaf_spot",
    "Strawberry___Leaf_scorch"
]

assert os.path.exists(SOURCE_DIR), f"Source directory not found: {SOURCE_DIR}"
assert os.path.exists(TARGET_DIR), f"Target directory not found: {TARGET_DIR}"

TARGET_CLASSES = sorted([
    d for d in os.listdir(TARGET_DIR)
    if os.path.isdir(os.path.join(TARGET_DIR, d)) and not d.startswith(".")
])

print("Target classes:", TARGET_CLASSES)

# ================= SEED =================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

# ================= GRADIENT REVERSAL =================
@tf.custom_gradient
def grad_reverse(x, lambd):
    def grad(dy):
        lambd_cast = tf.cast(lambd, dy.dtype)
        return -lambd_cast * dy, None
    return x, grad

class GradientReversal(layers.Layer):
    def __init__(self):
        super().__init__()
        self.lambd = tf.Variable(0.0, trainable=False, dtype=tf.float32)

    def call(self, x):
        return grad_reverse(x, self.lambd)

# ================= AUGMENTATION =================
def strong_aug_batch(x):
    x = tf.image.random_brightness(x, 0.45)
    x = tf.image.random_contrast(x, 0.65, 1.35)
    x = tf.image.random_flip_left_right(x)
    x = tf.image.random_flip_up_down(x)
    return tf.clip_by_value(x, 0, 1)

def weak_aug_batch(x):
    x = tf.image.random_brightness(x, 0.25)
    x = tf.image.random_contrast(x, 0.8, 1.2)
    x = tf.image.random_flip_left_right(x)
    return tf.clip_by_value(x, 0, 1)

# ================= GENERATOR =================
def combined_generator():
    datagen = ImageDataGenerator()

    s_gen = datagen.flow_from_directory(
        SOURCE_DIR,
        classes=SOURCE_CLASSES,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE // 2,
        shuffle=True,
        class_mode=None
    )

    t_gen = datagen.flow_from_directory(
        TARGET_DIR,
        classes=TARGET_CLASSES,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE // 2,
        shuffle=True,
        class_mode=None
    )

    while True:
        xs = next(s_gen)
        xt = next(t_gen)

        xs = tf.convert_to_tensor(xs, dtype=tf.float32) / 255.0
        xt = tf.convert_to_tensor(xt, dtype=tf.float32) / 255.0

        v1_s = strong_aug_batch(xs)
        v2_s = strong_aug_batch(xs)

        v1_t = weak_aug_batch(xt)
        v2_t = weak_aug_batch(xt)

        v1 = tf.concat([v1_s, v1_t], axis=0)
        v2 = tf.concat([v2_s, v2_t], axis=0)

        domain = tf.concat([
            tf.zeros(tf.shape(xs)[0], dtype=tf.int32),
            tf.ones(tf.shape(xt)[0], dtype=tf.int32)
        ], axis=0)

        yield (v1, v2), domain

# ================= SE-RESNET ENCODER =================
def se_block(x, ratio=8):
    filters = x.shape[-1]

    se = layers.GlobalAveragePooling2D()(x)
    se = layers.Dense(max(filters // ratio, 8), activation='relu')(se)
    se = layers.Dense(filters, activation='sigmoid')(se)
    se = layers.Reshape((1, 1, filters))(se)

    return layers.multiply([x, se])

def residual_block(x, filters, stride=1):
    shortcut = x

    x = layers.DepthwiseConv2D(3, strides=stride, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(filters, 1, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)

    x = se_block(x)

    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride, padding='same', use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    x = layers.Add()([x, shortcut])
    return layers.ReLU()(x)

def build_encoder():
    inp = layers.Input((IMG_SIZE, IMG_SIZE, 3))

    x = layers.Conv2D(16, 3, padding='same', use_bias=False)(inp)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    for f, s in zip([32, 32, 64, 64, 128, 128], [1, 1, 2, 1, 2, 1]):
        x = residual_block(x, f, s)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(FEATURE_DIM, activation='relu', name='shared_features')(x)

    return models.Model(inp, x, name="encoder_agronet")

def projection_head():
    inp = layers.Input((FEATURE_DIM,))
    x = layers.Dense(64, activation='relu')(inp)
    out = layers.Dense(PROJECTION_DIM)(x)
    return models.Model(inp, out, name="projection_head")

def domain_head(grl):
    inp = layers.Input((FEATURE_DIM,))
    x = grl(inp)
    x = layers.Dense(64, activation='relu')(x)
    out = layers.Dense(NUM_DOMAINS, activation='softmax')(x)
    return models.Model(inp, out, name="domain_head")

# ================= NT-XENT LOSS =================
def nt_xent(z1, z2):
    z1 = tf.cast(z1, tf.float32)
    z2 = tf.cast(z2, tf.float32)

    z1 = tf.math.l2_normalize(z1, axis=1)
    z2 = tf.math.l2_normalize(z2, axis=1)

    z = tf.concat([z1, z2], axis=0)
    sim = tf.matmul(z, z, transpose_b=True)

    mask = tf.eye(tf.shape(z)[0], dtype=sim.dtype) * tf.constant(1e9, dtype=sim.dtype)
    sim = sim - mask

    bs = tf.shape(z1)[0]
    labels = tf.concat([tf.range(bs, 2 * bs), tf.range(bs)], axis=0)

    logits = sim / tf.constant(TEMPERATURE, dtype=sim.dtype)

    loss = tf.keras.losses.sparse_categorical_crossentropy(
        labels,
        logits,
        from_logits=True
    )

    return tf.reduce_mean(loss)

# ================= MODEL =================
class SimCLR_MADALite(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.encoder = build_encoder()
        self.projector = projection_head()
        self.grl = GradientReversal()
        self.domain = domain_head(self.grl)

    def train_step(self, data):
        (v1, v2), dom = data

        with tf.GradientTape() as tape:
            h1 = self.encoder(v1, training=True)
            h2 = self.encoder(v2, training=True)

            z1 = self.projector(h1, training=True)
            z2 = self.projector(h2, training=True)

            contrastive_loss = nt_xent(z1, z2)

            h_all = tf.concat([h1, h2], axis=0)
            dom_all = tf.concat([dom, dom], axis=0)

            dom_pred = self.domain(h_all, training=True)

            domain_loss = tf.reduce_mean(
                tf.keras.losses.sparse_categorical_crossentropy(
                    dom_all,
                    dom_pred
                )
            )

            total_loss = contrastive_loss + DOMAIN_LOSS_WEIGHT * domain_loss

        grads = tape.gradient(total_loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))

        return {
            "contrastive_loss": contrastive_loss,
            "domain_loss": domain_loss,
            "total_loss": total_loss
        }

# ================= TARGET DATA =================
target_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

def get_target_generators(seed):
    train_gen = target_datagen.flow_from_directory(
        TARGET_DIR,
        classes=TARGET_CLASSES,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        subset='training',
        class_mode='categorical',
        shuffle=True,
        seed=seed
    )

    val_gen = target_datagen.flow_from_directory(
        TARGET_DIR,
        classes=TARGET_CLASSES,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        subset='validation',
        class_mode='categorical',
        shuffle=False,
        seed=seed
    )

    return train_gen, val_gen

# ================= EXPERIMENT =================
all_acc = []
all_latency = []

best_acc = -1
best_classifier = None
best_encoder = None
best_seed = None

for seed in SEEDS:
    print("\n" + "=" * 60)
    print(f"RUNNING SEED: {seed}")
    print("=" * 60)

    set_seed(seed)

    # Phase 1: Unsupervised training
    da_model = SimCLR_MADALite()
    da_model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        run_eagerly=False
    )

    gen = combined_generator()

    print("\nPhase 1: Unsupervised SimCLR + MADA-Lite Training")

    for epoch in range(EPOCHS_DA):
        da_model.grl.lambd.assign(min(1.0, epoch / 10.0))

        print(f"\nEpoch {epoch + 1}/{EPOCHS_DA} | GRL Lambda = {da_model.grl.lambd.numpy():.2f}")

        da_model.fit(
            gen,
            steps_per_epoch=STEPS_PER_EPOCH,
            epochs=1,
            verbose=1
        )

    # Phase 2: Frozen encoder evaluation
    print("\nPhase 2: Frozen Encoder + Classifier Evaluation")

    train_gen, val_gen = get_target_generators(seed)
    num_classes = len(TARGET_CLASSES)

    y_train = train_gen.classes
    class_ids = np.unique(y_train)

    cw_values = compute_class_weight(
        class_weight='balanced',
        classes=class_ids,
        y=y_train
    )

    class_weights = {i: 1.0 for i in range(num_classes)}
    for cls, w in zip(class_ids, cw_values):
        class_weights[int(cls)] = float(w)

    da_model.encoder.trainable = False

    classifier = tf.keras.Sequential([
        layers.Input((IMG_SIZE, IMG_SIZE, 3)),
        da_model.encoder,
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.25),
        layers.Dense(num_classes, activation='softmax')
    ])

    classifier.compile(
        optimizer=tf.keras.optimizers.Adam(5e-4),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        metrics=['accuracy']
    )

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_accuracy',
            patience=5,
            restore_best_weights=True,
            mode='max',
            verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_accuracy',
            factor=0.5,
            patience=3,
            min_lr=1e-5,
            mode='max',
            verbose=1
        )
    ]

    train_gen.reset()
    val_gen.reset()

    classifier.fit(
        train_gen,
        epochs=EPOCHS_LINEAR,
        validation_data=val_gen,
        class_weight=class_weights,
        callbacks=callbacks,
        verbose=1
    )

    val_gen.reset()
    loss, acc = classifier.evaluate(val_gen, verbose=0)

    print(f"\nSeed {seed} Target Accuracy: {acc * 100:.2f}%")
    all_acc.append(acc)

    # Latency
    val_gen.reset()
    sample_batch = next(val_gen)[0]

    _ = classifier.predict(sample_batch[:2], verbose=0)

    t0 = time.perf_counter()
    _ = classifier.predict(sample_batch, verbose=0)
    latency = (time.perf_counter() - t0) / len(sample_batch)

    all_latency.append(latency)

    print(f"Latency/Image: {latency * 1000:.2f} ms")

    if acc > best_acc:
        best_acc = acc
        best_classifier = classifier
        best_encoder = da_model.encoder
        best_seed = seed

# ================= SUMMARY =================
mean_acc = np.mean(all_acc)
std_acc = np.std(all_acc)
mean_latency = np.mean(all_latency)

print("\n" + "=" * 60)
print("MULTI-SEED SUMMARY")
print("=" * 60)

for seed, acc in zip(SEEDS, all_acc):
    print(f"Seed {seed}: {acc * 100:.2f}%")

print(f"\nMean Accuracy: {mean_acc * 100:.2f}%")
print(f"Std Accuracy : {std_acc * 100:.2f}%")
print(f"Best Accuracy: {best_acc * 100:.2f}%")
print(f"Best Seed    : {best_seed}")
print(f"Mean Latency : {mean_latency * 1000:.2f} ms/image")

# ================= CLASSIFICATION REPORT =================
print("\n" + "=" * 60)
print("BEST MODEL CLASSIFICATION REPORT")
print("=" * 60)

_, val_gen = get_target_generators(best_seed)

val_gen.reset()
y_true = val_gen.classes
y_pred = np.argmax(best_classifier.predict(val_gen, verbose=0), axis=1)

present_labels = sorted(np.unique(np.concatenate([y_true, y_pred])))
present_names = [TARGET_CLASSES[i] for i in present_labels]

print(
    classification_report(
        y_true,
        y_pred,
        labels=present_labels,
        target_names=present_names,
        zero_division=0
    )
)

cm = confusion_matrix(y_true, y_pred, labels=present_labels)

plt.figure(figsize=(8, 6))
plt.imshow(cm)
plt.title("Confusion Matrix - Frozen Encoder Evaluation")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(np.arange(len(present_names)), present_names, rotation=45, ha='right')
plt.yticks(np.arange(len(present_names)), present_names)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha='center', va='center')

plt.tight_layout()
plt.savefig("confusion_matrix_frozen_encoder_multiseed.png", dpi=150)
plt.show()

# ================= SAVE MODELS =================
best_encoder.save("encoder_frozen_agronet_multiseed.h5")
best_classifier.save("classifier_frozen_encoder_multiseed.h5")

encoder_size_kb = os.path.getsize("encoder_frozen_agronet_multiseed.h5") / 1024
full_size_kb = os.path.getsize("classifier_frozen_encoder_multiseed.h5") / 1024

params = best_classifier.count_params()
ram_fp32_mb = params * 4 / (1024 ** 2)

print("\n" + "=" * 60)
print("RESOURCE METRICS")
print("=" * 60)

print(f"Parameters       : {params:,}")
print(f"FP32 RAM Estimate: {ram_fp32_mb:.2f} MB")
print(f"Encoder Size     : {encoder_size_kb:.2f} KB")
print(f"Full Model Size  : {full_size_kb:.2f} KB")
print(f"Latency/Image    : {mean_latency * 1000:.2f} ms")
print(f"Throughput       : {1 / mean_latency:.2f} images/sec")
print(f"Energy Proxy     : {mean_latency * params:.2e}")

# ================= SAFE FLOAT32 TFLITE CONVERSION =================
tflite_file = "classifier_frozen_encoder_multiseed_float32.tflite"
tflite_size_kb = 0

try:
    converter = tf.lite.TFLiteConverter.from_keras_model(best_classifier)
    tflite_model = converter.convert()

    with open(tflite_file, "wb") as f:
        f.write(tflite_model)

    tflite_size_kb = os.path.getsize(tflite_file) / 1024
    print(f"\nFloat32 TFLite Size: {tflite_size_kb:.2f} KB")

except Exception as e:
    print("\nTFLite conversion failed, but training and evaluation completed.")
    print("Reason:", e)
    tflite_size_kb = 0

# ================= FINAL REPORT =================
print("\n" + "=" * 60)
print("FINAL CONSOLIDATED REPORT")
print("=" * 60)

print(f"Claim Type              : Unsupervised encoder + frozen downstream evaluation")
print(f"Source Classes Used     : {len(SOURCE_CLASSES)}")
print(f"Target Classes          : {TARGET_CLASSES}")
print(f"Image Size              : {IMG_SIZE}x{IMG_SIZE}")
print(f"DA Epochs               : {EPOCHS_DA}")
print(f"Steps per DA Epoch      : {STEPS_PER_EPOCH}")
print(f"Linear Classifier Epochs: {EPOCHS_LINEAR}")
print(f"Feature Dimension       : {FEATURE_DIM}")
print(f"Projection Dimension    : {PROJECTION_DIM}")
print(f"Domain Loss Weight      : {DOMAIN_LOSS_WEIGHT}")
print(f"Seeds                   : {SEEDS}")
print(f"Mean Accuracy ± Std     : {mean_acc * 100:.2f}% ± {std_acc * 100:.2f}%")
print(f"Best Accuracy           : {best_acc * 100:.2f}%")
print(f"Best Seed               : {best_seed}")
print(f"Encoder Size            : {encoder_size_kb:.2f} KB")
print(f"Full Model Size         : {full_size_kb:.2f} KB")
print(f"TFLite Size             : {tflite_size_kb:.2f} KB")
print(f"FP32 RAM Estimate       : {ram_fp32_mb:.2f} MB")
print(f"Inference Latency       : {mean_latency * 1000:.2f} ms/image")
print(f"Throughput              : {1 / mean_latency:.2f} images/sec")
print(f"Energy Proxy            : {mean_latency * params:.2e}")
print("=" * 60)

Multi-Seed running (9 Classes)

In [ ]:
# ===============================================================
# STABLE REVIEWER VERSION
# SimCLR + MADA-Lite + SE-ResNet
# Multi-seed + Frozen Encoder Evaluation + Safe TFLite
# ===============================================================

import os, random, time
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

# ================= STABLE FLOAT32 =================
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("float32")
print("Using float32 for stability.")

# ================= CONFIG =================
SOURCE_DIR = '/content/PlantVillage-Dataset/raw/segmented'
TARGET_DIR = '/content/RiceDiseases-DataSet'

IMG_SIZE = 48
BATCH_SIZE = 32

EPOCHS_DA = 20
STEPS_PER_EPOCH = 200
EPOCHS_LINEAR = 20

TEMPERATURE = 0.1
FEATURE_DIM = 64
PROJECTION_DIM = 64
NUM_DOMAINS = 2
DOMAIN_LOSS_WEIGHT = 0.10

SEEDS = [42, 123, 999]

SOURCE_CLASSES = [
    "Peach___Bacterial_spot",
    "Pepper,_bell___Bacterial_spot",
    "Tomato___Bacterial_spot",
    "Corn_(maize)___Northern_Leaf_Blight",
    "Potato___Early_blight",
    "Tomato___Target_Spot",
    "Corn_(maize)___Cercospora_leaf_spot_Gray_leaf_spot",
    "Tomato___Septoria_leaf_spot",
    "Strawberry___Leaf_scorch"
]

assert os.path.exists(SOURCE_DIR), f"Source directory not found: {SOURCE_DIR}"
assert os.path.exists(TARGET_DIR), f"Target directory not found: {TARGET_DIR}"

TARGET_CLASSES = sorted([
    d for d in os.listdir(TARGET_DIR)
    if os.path.isdir(os.path.join(TARGET_DIR, d)) and not d.startswith(".")
])

print("Target classes:", TARGET_CLASSES)

# ================= SEED =================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

# ================= GRADIENT REVERSAL =================
@tf.custom_gradient
def grad_reverse(x, lambd):
    def grad(dy):
        lambd_cast = tf.cast(lambd, dy.dtype)
        return -lambd_cast * dy, None
    return x, grad

class GradientReversal(layers.Layer):
    def __init__(self):
        super().__init__()
        self.lambd = tf.Variable(0.0, trainable=False, dtype=tf.float32)

    def call(self, x):
        return grad_reverse(x, self.lambd)

# ================= AUGMENTATION =================
def strong_aug_batch(x):
    x = tf.image.random_brightness(x, 0.45)
    x = tf.image.random_contrast(x, 0.65, 1.35)
    x = tf.image.random_flip_left_right(x)
    x = tf.image.random_flip_up_down(x)
    return tf.clip_by_value(x, 0, 1)

def weak_aug_batch(x):
    x = tf.image.random_brightness(x, 0.25)
    x = tf.image.random_contrast(x, 0.8, 1.2)
    x = tf.image.random_flip_left_right(x)
    return tf.clip_by_value(x, 0, 1)

# ================= GENERATOR =================
def combined_generator():
    datagen = ImageDataGenerator()

    s_gen = datagen.flow_from_directory(
        SOURCE_DIR,
        classes=SOURCE_CLASSES,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE // 2,
        shuffle=True,
        class_mode=None
    )

    t_gen = datagen.flow_from_directory(
        TARGET_DIR,
        classes=TARGET_CLASSES,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE // 2,
        shuffle=True,
        class_mode=None
    )

    while True:
        xs = next(s_gen)
        xt = next(t_gen)

        xs = tf.convert_to_tensor(xs, dtype=tf.float32) / 255.0
        xt = tf.convert_to_tensor(xt, dtype=tf.float32) / 255.0

        v1_s = strong_aug_batch(xs)
        v2_s = strong_aug_batch(xs)

        v1_t = weak_aug_batch(xt)
        v2_t = weak_aug_batch(xt)

        v1 = tf.concat([v1_s, v1_t], axis=0)
        v2 = tf.concat([v2_s, v2_t], axis=0)

        domain = tf.concat([
            tf.zeros(tf.shape(xs)[0], dtype=tf.int32),
            tf.ones(tf.shape(xt)[0], dtype=tf.int32)
        ], axis=0)

        yield (v1, v2), domain

# ================= SE-RESNET ENCODER =================
def se_block(x, ratio=8):
    filters = x.shape[-1]

    se = layers.GlobalAveragePooling2D()(x)
    se = layers.Dense(max(filters // ratio, 8), activation='relu')(se)
    se = layers.Dense(filters, activation='sigmoid')(se)
    se = layers.Reshape((1, 1, filters))(se)

    return layers.multiply([x, se])

def residual_block(x, filters, stride=1):
    shortcut = x

    x = layers.DepthwiseConv2D(3, strides=stride, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(filters, 1, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)

    x = se_block(x)

    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride, padding='same', use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    x = layers.Add()([x, shortcut])
    return layers.ReLU()(x)

def build_encoder():
    inp = layers.Input((IMG_SIZE, IMG_SIZE, 3))

    x = layers.Conv2D(16, 3, padding='same', use_bias=False)(inp)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    for f, s in zip([32, 32, 64, 64, 128, 128], [1, 1, 2, 1, 2, 1]):
        x = residual_block(x, f, s)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(FEATURE_DIM, activation='relu', name='shared_features')(x)

    return models.Model(inp, x, name="encoder_agronet")

def projection_head():
    inp = layers.Input((FEATURE_DIM,))
    x = layers.Dense(64, activation='relu')(inp)
    out = layers.Dense(PROJECTION_DIM)(x)
    return models.Model(inp, out, name="projection_head")

def domain_head(grl):
    inp = layers.Input((FEATURE_DIM,))
    x = grl(inp)
    x = layers.Dense(64, activation='relu')(x)
    out = layers.Dense(NUM_DOMAINS, activation='softmax')(x)
    return models.Model(inp, out, name="domain_head")

# ================= NT-XENT LOSS =================
def nt_xent(z1, z2):
    z1 = tf.cast(z1, tf.float32)
    z2 = tf.cast(z2, tf.float32)

    z1 = tf.math.l2_normalize(z1, axis=1)
    z2 = tf.math.l2_normalize(z2, axis=1)

    z = tf.concat([z1, z2], axis=0)
    sim = tf.matmul(z, z, transpose_b=True)

    mask = tf.eye(tf.shape(z)[0], dtype=sim.dtype) * tf.constant(1e9, dtype=sim.dtype)
    sim = sim - mask

    bs = tf.shape(z1)[0]
    labels = tf.concat([tf.range(bs, 2 * bs), tf.range(bs)], axis=0)

    logits = sim / tf.constant(TEMPERATURE, dtype=sim.dtype)

    loss = tf.keras.losses.sparse_categorical_crossentropy(
        labels,
        logits,
        from_logits=True
    )

    return tf.reduce_mean(loss)

# ================= MODEL =================
class SimCLR_MADALite(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.encoder = build_encoder()
        self.projector = projection_head()
        self.grl = GradientReversal()
        self.domain = domain_head(self.grl)

    def train_step(self, data):
        (v1, v2), dom = data

        with tf.GradientTape() as tape:
            h1 = self.encoder(v1, training=True)
            h2 = self.encoder(v2, training=True)

            z1 = self.projector(h1, training=True)
            z2 = self.projector(h2, training=True)

            contrastive_loss = nt_xent(z1, z2)

            h_all = tf.concat([h1, h2], axis=0)
            dom_all = tf.concat([dom, dom], axis=0)

            dom_pred = self.domain(h_all, training=True)

            domain_loss = tf.reduce_mean(
                tf.keras.losses.sparse_categorical_crossentropy(
                    dom_all,
                    dom_pred
                )
            )

            total_loss = contrastive_loss + DOMAIN_LOSS_WEIGHT * domain_loss

        grads = tape.gradient(total_loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))

        return {
            "contrastive_loss": contrastive_loss,
            "domain_loss": domain_loss,
            "total_loss": total_loss
        }

# ================= TARGET DATA =================
target_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

def get_target_generators(seed):
    train_gen = target_datagen.flow_from_directory(
        TARGET_DIR,
        classes=TARGET_CLASSES,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        subset='training',
        class_mode='categorical',
        shuffle=True,
        seed=seed
    )

    val_gen = target_datagen.flow_from_directory(
        TARGET_DIR,
        classes=TARGET_CLASSES,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        subset='validation',
        class_mode='categorical',
        shuffle=False,
        seed=seed
    )

    return train_gen, val_gen

# ================= EXPERIMENT =================
all_acc = []
all_latency = []

best_acc = -1
best_classifier = None
best_encoder = None
best_seed = None

for seed in SEEDS:
    print("\n" + "=" * 60)
    print(f"RUNNING SEED: {seed}")
    print("=" * 60)

    set_seed(seed)

    # Phase 1: Unsupervised training
    da_model = SimCLR_MADALite()
    da_model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        run_eagerly=False
    )

    gen = combined_generator()

    print("\nPhase 1: Unsupervised SimCLR + MADA-Lite Training")

    for epoch in range(EPOCHS_DA):
        da_model.grl.lambd.assign(min(1.0, epoch / 10.0))

        print(f"\nEpoch {epoch + 1}/{EPOCHS_DA} | GRL Lambda = {da_model.grl.lambd.numpy():.2f}")

        da_model.fit(
            gen,
            steps_per_epoch=STEPS_PER_EPOCH,
            epochs=1,
            verbose=1
        )

    # Phase 2: Frozen encoder evaluation
    print("\nPhase 2: Frozen Encoder + Classifier Evaluation")

    train_gen, val_gen = get_target_generators(seed)
    num_classes = len(TARGET_CLASSES)

    y_train = train_gen.classes
    class_ids = np.unique(y_train)

    cw_values = compute_class_weight(
        class_weight='balanced',
        classes=class_ids,
        y=y_train
    )

    class_weights = {i: 1.0 for i in range(num_classes)}
    for cls, w in zip(class_ids, cw_values):
        class_weights[int(cls)] = float(w)

    da_model.encoder.trainable = False

    classifier = tf.keras.Sequential([
        layers.Input((IMG_SIZE, IMG_SIZE, 3)),
        da_model.encoder,
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.25),
        layers.Dense(num_classes, activation='softmax')
    ])

    classifier.compile(
        optimizer=tf.keras.optimizers.Adam(5e-4),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        metrics=['accuracy']
    )

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_accuracy',
            patience=5,
            restore_best_weights=True,
            mode='max',
            verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_accuracy',
            factor=0.5,
            patience=3,
            min_lr=1e-5,
            mode='max',
            verbose=1
        )
    ]

    train_gen.reset()
    val_gen.reset()

    classifier.fit(
        train_gen,
        epochs=EPOCHS_LINEAR,
        validation_data=val_gen,
        class_weight=class_weights,
        callbacks=callbacks,
        verbose=1
    )

    val_gen.reset()
    loss, acc = classifier.evaluate(val_gen, verbose=0)

    print(f"\nSeed {seed} Target Accuracy: {acc * 100:.2f}%")
    all_acc.append(acc)

    # Latency
    val_gen.reset()
    sample_batch = next(val_gen)[0]

    _ = classifier.predict(sample_batch[:2], verbose=0)

    t0 = time.perf_counter()
    _ = classifier.predict(sample_batch, verbose=0)
    latency = (time.perf_counter() - t0) / len(sample_batch)

    all_latency.append(latency)

    print(f"Latency/Image: {latency * 1000:.2f} ms")

    if acc > best_acc:
        best_acc = acc
        best_classifier = classifier
        best_encoder = da_model.encoder
        best_seed = seed

# ================= SUMMARY =================
mean_acc = np.mean(all_acc)
std_acc = np.std(all_acc)
mean_latency = np.mean(all_latency)

print("\n" + "=" * 60)
print("MULTI-SEED SUMMARY")
print("=" * 60)

for seed, acc in zip(SEEDS, all_acc):
    print(f"Seed {seed}: {acc * 100:.2f}%")

print(f"\nMean Accuracy: {mean_acc * 100:.2f}%")
print(f"Std Accuracy : {std_acc * 100:.2f}%")
print(f"Best Accuracy: {best_acc * 100:.2f}%")
print(f"Best Seed    : {best_seed}")
print(f"Mean Latency : {mean_latency * 1000:.2f} ms/image")

# ================= CLASSIFICATION REPORT =================
print("\n" + "=" * 60)
print("BEST MODEL CLASSIFICATION REPORT")
print("=" * 60)

_, val_gen = get_target_generators(best_seed)

val_gen.reset()
y_true = val_gen.classes
y_pred = np.argmax(best_classifier.predict(val_gen, verbose=0), axis=1)

present_labels = sorted(np.unique(np.concatenate([y_true, y_pred])))
present_names = [TARGET_CLASSES[i] for i in present_labels]

print(
    classification_report(
        y_true,
        y_pred,
        labels=present_labels,
        target_names=present_names,
        zero_division=0
    )
)

cm = confusion_matrix(y_true, y_pred, labels=present_labels)

plt.figure(figsize=(8, 6))
plt.imshow(cm)
plt.title("Confusion Matrix - Frozen Encoder Evaluation")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(np.arange(len(present_names)), present_names, rotation=45, ha='right')
plt.yticks(np.arange(len(present_names)), present_names)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha='center', va='center')

plt.tight_layout()
plt.savefig("confusion_matrix_frozen_encoder_multiseed.png", dpi=150)
plt.show()

# ================= SAVE MODELS =================
best_encoder.save("encoder_frozen_agronet_multiseed.h5")
best_classifier.save("classifier_frozen_encoder_multiseed.h5")

encoder_size_kb = os.path.getsize("encoder_frozen_agronet_multiseed.h5") / 1024
full_size_kb = os.path.getsize("classifier_frozen_encoder_multiseed.h5") / 1024

params = best_classifier.count_params()
ram_fp32_mb = params * 4 / (1024 ** 2)

print("\n" + "=" * 60)
print("RESOURCE METRICS")
print("=" * 60)

print(f"Parameters       : {params:,}")
print(f"FP32 RAM Estimate: {ram_fp32_mb:.2f} MB")
print(f"Encoder Size     : {encoder_size_kb:.2f} KB")
print(f"Full Model Size  : {full_size_kb:.2f} KB")
print(f"Latency/Image    : {mean_latency * 1000:.2f} ms")
print(f"Throughput       : {1 / mean_latency:.2f} images/sec")
print(f"Energy Proxy     : {mean_latency * params:.2e}")

# ================= SAFE FLOAT32 TFLITE CONVERSION =================
tflite_file = "classifier_frozen_encoder_multiseed_float32.tflite"
tflite_size_kb = 0

try:
    converter = tf.lite.TFLiteConverter.from_keras_model(best_classifier)
    tflite_model = converter.convert()

    with open(tflite_file, "wb") as f:
        f.write(tflite_model)

    tflite_size_kb = os.path.getsize(tflite_file) / 1024
    print(f"\nFloat32 TFLite Size: {tflite_size_kb:.2f} KB")

except Exception as e:
    print("\nTFLite conversion failed, but training and evaluation completed.")
    print("Reason:", e)
    tflite_size_kb = 0

# ================= FINAL REPORT =================
print("\n" + "=" * 60)
print("FINAL CONSOLIDATED REPORT")
print("=" * 60)

print(f"Claim Type              : Unsupervised encoder + frozen downstream evaluation")
print(f"Source Classes Used     : {len(SOURCE_CLASSES)}")
print(f"Target Classes          : {TARGET_CLASSES}")
print(f"Image Size              : {IMG_SIZE}x{IMG_SIZE}")
print(f"DA Epochs               : {EPOCHS_DA}")
print(f"Steps per DA Epoch      : {STEPS_PER_EPOCH}")
print(f"Linear Classifier Epochs: {EPOCHS_LINEAR}")
print(f"Feature Dimension       : {FEATURE_DIM}")
print(f"Projection Dimension    : {PROJECTION_DIM}")
print(f"Domain Loss Weight      : {DOMAIN_LOSS_WEIGHT}")
print(f"Seeds                   : {SEEDS}")
print(f"Mean Accuracy ± Std     : {mean_acc * 100:.2f}% ± {std_acc * 100:.2f}%")
print(f"Best Accuracy           : {best_acc * 100:.2f}%")
print(f"Best Seed               : {best_seed}")
print(f"Encoder Size            : {encoder_size_kb:.2f} KB")
print(f"Full Model Size         : {full_size_kb:.2f} KB")
print(f"TFLite Size             : {tflite_size_kb:.2f} KB")
print(f"FP32 RAM Estimate       : {ram_fp32_mb:.2f} MB")
print(f"Inference Latency       : {mean_latency * 1000:.2f} ms/image")
print(f"Throughput              : {1 / mean_latency:.2f} images/sec")
print(f"Energy Proxy            : {mean_latency * params:.2e}")
print("=" * 60)

In [ ]:
# ===============================================================
# SimCLR + MADA-lite + SE-ResNet + INT8 Quantization (9 classes)
# ===============================================================

import os, numpy as np, tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# ================= CONFIG =================
IMG_SIZE = 48
BATCH_SIZE = 32
EPOCHS_DA = 20
TEMPERATURE = 0.1

FEATURE_DIM = 64       # reduced
PROJECTION_DIM = 64    # reduced
NUM_DOMAINS = 2

SOURCE_DIR = '/content/PlantVillage-Dataset/raw/segmented'
TARGET_DIR = '/content/RiceDiseases-DataSet'

SOURCE_CLASSES = [
    "Peach___Bacterial_spot",
    "Pepper,_bell___Bacterial_spot",
    "Tomato___Bacterial_spot",
    "Corn_(maize)___Northern_Leaf_Blight",
    "Potato___Early_blight",
    "Tomato___Target_Spot",
    "Corn_(maize)___Cercospora_leaf_spot_Gray_leaf_spot",
    "Tomato___Septoria_leaf_spot",
    "Strawberry___Leaf_scorch"
]

TARGET_CLASSES = sorted([d for d in os.listdir(TARGET_DIR) if os.path.isdir(os.path.join(TARGET_DIR, d))])

# ================= GRADIENT REVERSAL =================
@tf.custom_gradient
def grad_reverse(x, lambd):
    def grad(dy):
        return -lambd * dy, None
    return x, grad

class GradientReversal(layers.Layer):
    def __init__(self):
        super().__init__()
        self.lambd = tf.Variable(0.0, trainable=False, dtype=tf.float32)
    def call(self, x):
        return grad_reverse(x, self.lambd)

# ================= AUGMENTATION =================
def strong_aug(x):
    x = tf.image.random_brightness(x, 0.5)
    x = tf.image.random_contrast(x, 0.6, 1.4)
    x = tf.image.random_flip_left_right(x)
    return tf.clip_by_value(x, 0, 1)

def weak_aug(x):
    x = tf.image.random_brightness(x, 0.3)
    x = tf.image.random_contrast(x, 0.8, 1.2)
    return tf.clip_by_value(x, 0, 1)

def create_views(img, is_source):
    img = tf.cast(img, tf.float32) / 255.0
    aug = strong_aug if is_source else weak_aug
    return aug(img), aug(img)

# ================= DATA GENERATOR =================
def combined_generator():
    datagen = ImageDataGenerator()
    s_gen = datagen.flow_from_directory(
        SOURCE_DIR, classes=SOURCE_CLASSES,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE // 2, shuffle=True, class_mode=None
    )
    t_gen = datagen.flow_from_directory(
        TARGET_DIR,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE // 2, shuffle=True, class_mode=None
    )
    while True:
        xs = next(s_gen)
        xt = next(t_gen)
        X = np.concatenate([xs, xt], axis=0)
        domain = tf.concat([tf.zeros(xs.shape[0], dtype=tf.int32),
                            tf.ones(xt.shape[0], dtype=tf.int32)], axis=0)
        v1, v2 = [], []
        for i, img in enumerate(X):
            a, b = create_views(img, i < xs.shape[0])
            v1.append(a)
            v2.append(b)
        yield (tf.stack(v1), tf.stack(v2)), domain

# ================= SE-RESNET BACKBONE =================
def se_block(x, ratio=8):
    filters = x.shape[-1]
    se = layers.GlobalAveragePooling2D()(x)
    se = layers.Dense(filters // ratio, activation='relu')(se)
    se = layers.Dense(filters, activation='sigmoid')(se)
    se = layers.Reshape((1,1,filters))(se)
    return layers.multiply([x, se])

def residual_block(x, filters, stride=1):
    shortcut = x
    x = layers.DepthwiseConv2D(3, strides=stride, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(filters, 1, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = se_block(x)
    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride, padding='same')(shortcut)
    x = layers.Add()([x, shortcut])
    return layers.ReLU()(x)

def build_encoder():
    inp = layers.Input((IMG_SIZE, IMG_SIZE, 3))
    x = layers.Conv2D(16, 3, padding='same')(inp)  # reduced filters
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    for f, s in zip([32, 32, 64, 64, 128, 128], [1,1,2,1,2,1]):  # reduced filters
        x = residual_block(x, f, s)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(FEATURE_DIM, activation='relu', name='shared_features')(x)
    return models.Model(inp, x, name="encoder_agronet")

def projection_head():
    inp = layers.Input((FEATURE_DIM,))
    x = layers.Dense(32, activation='relu')(inp)
    out = layers.Dense(PROJECTION_DIM)(x)
    return models.Model(inp, out, name="projection")

def domain_head(grl):
    inp = layers.Input((FEATURE_DIM,))
    x = grl(inp)
    x = layers.Dense(32, activation='relu')(x)
    out = layers.Dense(NUM_DOMAINS, activation='softmax')(x)
    return models.Model(inp, out, name="domain")

# ================= NT-XENT LOSS =================
def nt_xent(z1, z2):
    z1 = tf.math.l2_normalize(z1, axis=1)
    z2 = tf.math.l2_normalize(z2, axis=1)
    z = tf.concat([z1, z2], axis=0)
    sim = tf.matmul(z, z, transpose_b=True)
    sim -= tf.eye(tf.shape(z)[0]) * 1e9
    bs = tf.shape(z1)[0]
    labels = tf.concat([tf.range(bs, 2*bs), tf.range(bs)], axis=0)
    loss = tf.keras.losses.sparse_categorical_crossentropy(labels, sim / TEMPERATURE, from_logits=True)
    return tf.reduce_mean(loss)

# ================= SIMCLR + MADA-LITE MODEL =================
class SimCLR_DA(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.encoder = build_encoder()
        self.projector = projection_head()
        self.grl = GradientReversal()
        self.domain = domain_head(self.grl)

    def train_step(self, data):
        (v1, v2), dom = data
        with tf.GradientTape() as tape:
            h1 = self.encoder(v1, training=True)
            h2 = self.encoder(v2, training=True)
            z1 = self.projector(h1, training=True)
            z2 = self.projector(h2, training=True)
            loss_c = nt_xent(z1, z2)
            pseudo = tf.stop_gradient(tf.nn.softmax(tf.concat([h1,h2], axis=0)))
            weights = tf.reduce_max(pseudo, axis=1)
            dom_pred = self.domain(tf.concat([h1,h2], axis=0), training=True)
            loss_d = tf.reduce_mean(
                tf.keras.losses.sparse_categorical_crossentropy(
                    tf.concat([dom, dom], axis=0), dom_pred
                ) * weights
            )
            loss = loss_c + loss_d
        grads = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        return {"contrastive_loss": loss_c, "domain_loss": loss_d}

# ================= PHASE 1: DOMAIN ADAPTATION =================
model = SimCLR_DA()
model.compile(optimizer=tf.keras.optimizers.Adam(1e-4))
gen = combined_generator()
print("\n Training SimCLR + MADA-lite DA")
for epoch in range(EPOCHS_DA):
    model.grl.lambd.assign(min(1.0, epoch / 10.0))
    print(f"Epoch {epoch+1} | DOMAIN_LAMBDA = {model.grl.lambd.numpy():.2f}")
    model.fit(gen, steps_per_epoch=200, epochs=1, verbose=1)
model.encoder.save("encoder_da_agronet.h5")

# ================= PHASE 2: LINEAR EVALUATION =================
model.encoder.trainable = False
classifier = tf.keras.Sequential([
    layers.Input((IMG_SIZE, IMG_SIZE, 3)),
    model.encoder,
    layers.Dense(len(TARGET_CLASSES), activation='softmax')
])
classifier.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                   loss='categorical_crossentropy',
                   metrics=['accuracy'])

datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)
train_gen = datagen.flow_from_directory(
    TARGET_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=32, subset='training', class_mode='categorical'
)
val_gen = datagen.flow_from_directory(
    TARGET_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=32, subset='validation', class_mode='categorical', shuffle=False
)

print("\n Training Linear Classifier")
classifier.fit(train_gen, epochs=20, validation_data=val_gen, verbose=1)
loss, acc = classifier.evaluate(val_gen, verbose=0)

# ================= RESULTS =================
classifier.save("full_da_model_agronet.h5")
encoder_size = os.path.getsize("encoder_da_agronet.h5") / 1024  # KB
full_size = os.path.getsize("full_da_model_agronet.h5") / 1024  # KB
print("\n==============================")
print(f"Target Accuracy : {acc*100:.2f}%")
print(f" Encoder Size   : {encoder_size:.2f} KB")
print(f" Full Model Size: {full_size:.2f} KB")
print("==============================")

# ================= TFLITE CONVERSION + INT8 QUANT =================
def representative_data_gen():
    for images, _ in train_gen:
        yield [images.astype(np.float32)]
        break  # only one batch is enough

tflite_model_file = "full_da_model_agronet_int8_small.tflite"
converter = tf.lite.TFLiteConverter.from_keras_model(classifier)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8
tflite_model = converter.convert()
with open(tflite_model_file, "wb") as f:
    f.write(tflite_model)
tflite_size = os.path.getsize(tflite_model_file)  # bytes
print(f" INT8 TFLite Model Size: {tflite_size/1024:.2f} KB")

Multi seed run (All 38 classess)




In [ ]:
# ===============================================================
# STABLE REVIEWER VERSION
# SimCLR + MADA-Lite + SE-ResNet
# Multi-seed + Frozen Encoder Evaluation + Safe TFLite
# SOURCE DOMAIN: FULL PLANTVILLAGE 38 CLASSES
# ===============================================================

import os, random, time
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

# ================= STABLE FLOAT32 =================
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("float32")
print("Using float32 for stability.")

# ================= CONFIG =================
SOURCE_DIR = '/content/PlantVillage-Dataset/raw/segmented'
TARGET_DIR = '/content/RiceDiseases-DataSet'

IMG_SIZE = 48
BATCH_SIZE = 32

EPOCHS_DA = 20
STEPS_PER_EPOCH = 200
EPOCHS_LINEAR = 20

TEMPERATURE = 0.1
FEATURE_DIM = 64
PROJECTION_DIM = 64
NUM_DOMAINS = 2
DOMAIN_LOSS_WEIGHT = 0.10

SEEDS = [42, 123, 999]

assert os.path.exists(SOURCE_DIR), f"Source directory not found: {SOURCE_DIR}"
assert os.path.exists(TARGET_DIR), f"Target directory not found: {TARGET_DIR}"

# ================= SOURCE CLASSES: FULL 38 PLANTVILLAGE =================
SOURCE_CLASSES = sorted([
    d for d in os.listdir(SOURCE_DIR)
    if os.path.isdir(os.path.join(SOURCE_DIR, d))
    and not d.startswith(".")
])

print("\n" + "=" * 60)
print("SOURCE DOMAIN INFORMATION")
print("=" * 60)

for cls in SOURCE_CLASSES:
    print(cls)

print(f"\nTotal Source Classes: {len(SOURCE_CLASSES)}")

assert len(SOURCE_CLASSES) == 38, \
    f"Expected 38 PlantVillage classes, but found {len(SOURCE_CLASSES)} classes."

# ================= TARGET CLASSES =================
TARGET_CLASSES = sorted([
    d for d in os.listdir(TARGET_DIR)
    if os.path.isdir(os.path.join(TARGET_DIR, d))
    and not d.startswith(".")
])

print("\nTarget classes:", TARGET_CLASSES)

# ================= SEED =================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

# ================= GRADIENT REVERSAL =================
@tf.custom_gradient
def grad_reverse(x, lambd):
    def grad(dy):
        lambd_cast = tf.cast(lambd, dy.dtype)
        return -lambd_cast * dy, None
    return x, grad

class GradientReversal(layers.Layer):
    def __init__(self):
        super().__init__()
        self.lambd = tf.Variable(0.0, trainable=False, dtype=tf.float32)

    def call(self, x):
        return grad_reverse(x, self.lambd)

# ================= AUGMENTATION =================
def strong_aug_batch(x):
    x = tf.image.random_brightness(x, 0.45)
    x = tf.image.random_contrast(x, 0.65, 1.35)
    x = tf.image.random_flip_left_right(x)
    x = tf.image.random_flip_up_down(x)
    return tf.clip_by_value(x, 0, 1)

def weak_aug_batch(x):
    x = tf.image.random_brightness(x, 0.25)
    x = tf.image.random_contrast(x, 0.8, 1.2)
    x = tf.image.random_flip_left_right(x)
    return tf.clip_by_value(x, 0, 1)

# ================= GENERATOR =================
def combined_generator(seed=None):
    datagen = ImageDataGenerator()

    s_gen = datagen.flow_from_directory(
        SOURCE_DIR,
        classes=SOURCE_CLASSES,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE // 2,
        shuffle=True,
        class_mode=None,
        seed=seed
    )

    t_gen = datagen.flow_from_directory(
        TARGET_DIR,
        classes=TARGET_CLASSES,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE // 2,
        shuffle=True,
        class_mode=None,
        seed=seed
    )

    while True:
        xs = next(s_gen)
        xt = next(t_gen)

        xs = tf.convert_to_tensor(xs, dtype=tf.float32) / 255.0
        xt = tf.convert_to_tensor(xt, dtype=tf.float32) / 255.0

        v1_s = strong_aug_batch(xs)
        v2_s = strong_aug_batch(xs)

        v1_t = weak_aug_batch(xt)
        v2_t = weak_aug_batch(xt)

        v1 = tf.concat([v1_s, v1_t], axis=0)
        v2 = tf.concat([v2_s, v2_t], axis=0)

        domain = tf.concat([
            tf.zeros(tf.shape(xs)[0], dtype=tf.int32),
            tf.ones(tf.shape(xt)[0], dtype=tf.int32)
        ], axis=0)

        yield (v1, v2), domain

# ================= SE-RESNET ENCODER =================
def se_block(x, ratio=8):
    filters = x.shape[-1]

    se = layers.GlobalAveragePooling2D()(x)
    se = layers.Dense(max(filters // ratio, 8), activation='relu')(se)
    se = layers.Dense(filters, activation='sigmoid')(se)
    se = layers.Reshape((1, 1, filters))(se)

    return layers.multiply([x, se])

def residual_block(x, filters, stride=1):
    shortcut = x

    x = layers.DepthwiseConv2D(3, strides=stride, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(filters, 1, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)

    x = se_block(x)

    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride, padding='same', use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    x = layers.Add()([x, shortcut])
    return layers.ReLU()(x)

def build_encoder():
    inp = layers.Input((IMG_SIZE, IMG_SIZE, 3))

    x = layers.Conv2D(16, 3, padding='same', use_bias=False)(inp)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    for f, s in zip([32, 32, 64, 64, 128, 128], [1, 1, 2, 1, 2, 1]):
        x = residual_block(x, f, s)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(FEATURE_DIM, activation='relu', name='shared_features')(x)

    return models.Model(inp, x, name="encoder_agronet")

def projection_head():
    inp = layers.Input((FEATURE_DIM,))
    x = layers.Dense(64, activation='relu')(inp)
    out = layers.Dense(PROJECTION_DIM)(x)
    return models.Model(inp, out, name="projection_head")

def domain_head(grl):
    inp = layers.Input((FEATURE_DIM,))
    x = grl(inp)
    x = layers.Dense(64, activation='relu')(x)
    out = layers.Dense(NUM_DOMAINS, activation='softmax')(x)
    return models.Model(inp, out, name="domain_head")

# ================= NT-XENT LOSS =================
def nt_xent(z1, z2):
    z1 = tf.cast(z1, tf.float32)
    z2 = tf.cast(z2, tf.float32)

    z1 = tf.math.l2_normalize(z1, axis=1)
    z2 = tf.math.l2_normalize(z2, axis=1)

    z = tf.concat([z1, z2], axis=0)
    sim = tf.matmul(z, z, transpose_b=True)

    mask = tf.eye(tf.shape(z)[0], dtype=sim.dtype) * tf.constant(1e9, dtype=sim.dtype)
    sim = sim - mask

    bs = tf.shape(z1)[0]
    labels = tf.concat([tf.range(bs, 2 * bs), tf.range(bs)], axis=0)

    logits = sim / tf.constant(TEMPERATURE, dtype=sim.dtype)

    loss = tf.keras.losses.sparse_categorical_crossentropy(
        labels,
        logits,
        from_logits=True
    )

    return tf.reduce_mean(loss)

# ================= MODEL =================
class SimCLR_MADALite(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.encoder = build_encoder()
        self.projector = projection_head()
        self.grl = GradientReversal()
        self.domain = domain_head(self.grl)

    def train_step(self, data):
        (v1, v2), dom = data

        with tf.GradientTape() as tape:
            h1 = self.encoder(v1, training=True)
            h2 = self.encoder(v2, training=True)

            z1 = self.projector(h1, training=True)
            z2 = self.projector(h2, training=True)

            contrastive_loss = nt_xent(z1, z2)

            h_all = tf.concat([h1, h2], axis=0)
            dom_all = tf.concat([dom, dom], axis=0)

            dom_pred = self.domain(h_all, training=True)

            domain_loss = tf.reduce_mean(
                tf.keras.losses.sparse_categorical_crossentropy(
                    dom_all,
                    dom_pred
                )
            )

            total_loss = contrastive_loss + DOMAIN_LOSS_WEIGHT * domain_loss

        grads = tape.gradient(total_loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))

        return {
            "contrastive_loss": contrastive_loss,
            "domain_loss": domain_loss,
            "total_loss": total_loss
        }

# ================= TARGET DATA =================
target_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

def get_target_generators(seed):
    train_gen = target_datagen.flow_from_directory(
        TARGET_DIR,
        classes=TARGET_CLASSES,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        subset='training',
        class_mode='categorical',
        shuffle=True,
        seed=seed
    )

    val_gen = target_datagen.flow_from_directory(
        TARGET_DIR,
        classes=TARGET_CLASSES,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        subset='validation',
        class_mode='categorical',
        shuffle=False,
        seed=seed
    )

    return train_gen, val_gen

# ================= EXPERIMENT =================
all_acc = []
all_latency = []

best_acc = -1
best_classifier = None
best_encoder = None
best_seed = None

for seed in SEEDS:
    print("\n" + "=" * 60)
    print(f"RUNNING SEED: {seed}")
    print("=" * 60)

    set_seed(seed)

    # Phase 1: Unsupervised training
    da_model = SimCLR_MADALite()
    da_model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        run_eagerly=False
    )

    gen = combined_generator(seed=seed)

    print("\nPhase 1: Unsupervised SimCLR + MADA-Lite Training")

    for epoch in range(EPOCHS_DA):
        da_model.grl.lambd.assign(min(1.0, epoch / 10.0))

        print(f"\nEpoch {epoch + 1}/{EPOCHS_DA} | GRL Lambda = {da_model.grl.lambd.numpy():.2f}")

        da_model.fit(
            gen,
            steps_per_epoch=STEPS_PER_EPOCH,
            epochs=1,
            verbose=1
        )

    # Phase 2: Frozen encoder evaluation
    print("\nPhase 2: Frozen Encoder + Classifier Evaluation")

    train_gen, val_gen = get_target_generators(seed)
    num_classes = len(TARGET_CLASSES)

    y_train = train_gen.classes
    class_ids = np.unique(y_train)

    cw_values = compute_class_weight(
        class_weight='balanced',
        classes=class_ids,
        y=y_train
    )

    class_weights = {i: 1.0 for i in range(num_classes)}
    for cls, w in zip(class_ids, cw_values):
        class_weights[int(cls)] = float(w)

    da_model.encoder.trainable = False

    classifier = tf.keras.Sequential([
        layers.Input((IMG_SIZE, IMG_SIZE, 3)),
        da_model.encoder,
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.25),
        layers.Dense(num_classes, activation='softmax')
    ])

    classifier.compile(
        optimizer=tf.keras.optimizers.Adam(5e-4),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        metrics=['accuracy']
    )

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_accuracy',
            patience=5,
            restore_best_weights=True,
            mode='max',
            verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_accuracy',
            factor=0.5,
            patience=3,
            min_lr=1e-5,
            mode='max',
            verbose=1
        )
    ]

    train_gen.reset()
    val_gen.reset()

    classifier.fit(
        train_gen,
        epochs=EPOCHS_LINEAR,
        validation_data=val_gen,
        class_weight=class_weights,
        callbacks=callbacks,
        verbose=1
    )

    val_gen.reset()
    loss, acc = classifier.evaluate(val_gen, verbose=0)

    print(f"\nSeed {seed} Target Accuracy: {acc * 100:.2f}%")
    all_acc.append(acc)

    # Latency
    val_gen.reset()
    sample_batch = next(val_gen)[0]

    _ = classifier.predict(sample_batch[:2], verbose=0)

    t0 = time.perf_counter()
    _ = classifier.predict(sample_batch, verbose=0)
    latency = (time.perf_counter() - t0) / len(sample_batch)

    all_latency.append(latency)

    print(f"Latency/Image: {latency * 1000:.2f} ms")

    if acc > best_acc:
        best_acc = acc
        best_classifier = classifier
        best_encoder = da_model.encoder
        best_seed = seed

# ================= SUMMARY =================
mean_acc = np.mean(all_acc)
std_acc = np.std(all_acc)
mean_latency = np.mean(all_latency)

print("\n" + "=" * 60)
print("MULTI-SEED SUMMARY")
print("=" * 60)

for seed, acc in zip(SEEDS, all_acc):
    print(f"Seed {seed}: {acc * 100:.2f}%")

print(f"\nMean Accuracy: {mean_acc * 100:.2f}%")
print(f"Std Accuracy : {std_acc * 100:.2f}%")
print(f"Best Accuracy: {best_acc * 100:.2f}%")
print(f"Best Seed    : {best_seed}")
print(f"Mean Latency : {mean_latency * 1000:.2f} ms/image")

# ================= CLASSIFICATION REPORT =================
print("\n" + "=" * 60)
print("BEST MODEL CLASSIFICATION REPORT")
print("=" * 60)

_, val_gen = get_target_generators(best_seed)

val_gen.reset()
y_true = val_gen.classes
y_pred = np.argmax(best_classifier.predict(val_gen, verbose=0), axis=1)

present_labels = sorted(np.unique(np.concatenate([y_true, y_pred])))
present_names = [TARGET_CLASSES[i] for i in present_labels]

print(
    classification_report(
        y_true,
        y_pred,
        labels=present_labels,
        target_names=present_names,
        zero_division=0
    )
)

cm = confusion_matrix(y_true, y_pred, labels=present_labels)

plt.figure(figsize=(8, 6))
plt.imshow(cm)
plt.title("Confusion Matrix - Frozen Encoder Evaluation")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(np.arange(len(present_names)), present_names, rotation=45, ha='right')
plt.yticks(np.arange(len(present_names)), present_names)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha='center', va='center')

plt.tight_layout()
plt.savefig("confusion_matrix_frozen_encoder_multiseed_38source.png", dpi=150)
plt.show()

# ================= SAVE MODELS =================
best_encoder.save("encoder_frozen_agronet_multiseed_38source.h5")
best_classifier.save("classifier_frozen_encoder_multiseed_38source.h5")

encoder_size_kb = os.path.getsize("encoder_frozen_agronet_multiseed_38source.h5") / 1024
full_size_kb = os.path.getsize("classifier_frozen_encoder_multiseed_38source.h5") / 1024

params = best_classifier.count_params()
ram_fp32_mb = params * 4 / (1024 ** 2)

print("\n" + "=" * 60)
print("RESOURCE METRICS")
print("=" * 60)

print(f"Parameters       : {params:,}")
print(f"FP32 RAM Estimate: {ram_fp32_mb:.2f} MB")
print(f"Encoder Size     : {encoder_size_kb:.2f} KB")
print(f"Full Model Size  : {full_size_kb:.2f} KB")
print(f"Latency/Image    : {mean_latency * 1000:.2f} ms")
print(f"Throughput       : {1 / mean_latency:.2f} images/sec")
print(f"Energy Proxy     : {mean_latency * params:.2e}")

# ================= SAFE FLOAT32 TFLITE CONVERSION =================
tflite_file = "classifier_frozen_encoder_multiseed_38source_float32.tflite"
tflite_size_kb = 0

try:
    converter = tf.lite.TFLiteConverter.from_keras_model(best_classifier)
    tflite_model = converter.convert()

    with open(tflite_file, "wb") as f:
        f.write(tflite_model)

    tflite_size_kb = os.path.getsize(tflite_file) / 1024
    print(f"\nFloat32 TFLite Size: {tflite_size_kb:.2f} KB")

except Exception as e:
    print("\nTFLite conversion failed, but training and evaluation completed.")
    print("Reason:", e)
    tflite_size_kb = 0

# ================= FINAL REPORT =================
print("\n" + "=" * 60)
print("FINAL CONSOLIDATED REPORT")
print("=" * 60)

print(f"Claim Type              : Unsupervised encoder + frozen downstream evaluation")
print(f"Source Domain           : PlantVillage full 38-class source domain")
print(f"Source Classes Used     : {len(SOURCE_CLASSES)}")
print(f"Target Classes          : {TARGET_CLASSES}")
print(f"Image Size              : {IMG_SIZE}x{IMG_SIZE}")
print(f"DA Epochs               : {EPOCHS_DA}")
print(f"Steps per DA Epoch      : {STEPS_PER_EPOCH}")
print(f"Linear Classifier Epochs: {EPOCHS_LINEAR}")
print(f"Feature Dimension       : {FEATURE_DIM}")
print(f"Projection Dimension    : {PROJECTION_DIM}")
print(f"Domain Loss Weight      : {DOMAIN_LOSS_WEIGHT}")
print(f"Seeds                   : {SEEDS}")
print(f"Mean Accuracy ± Std     : {mean_acc * 100:.2f}% ± {std_acc * 100:.2f}%")
print(f"Best Accuracy           : {best_acc * 100:.2f}%")
print(f"Best Seed               : {best_seed}")
print(f"Encoder Size            : {encoder_size_kb:.2f} KB")
print(f"Full Model Size         : {full_size_kb:.2f} KB")
print(f"TFLite Size             : {tflite_size_kb:.2f} KB")
print(f"FP32 RAM Estimate       : {ram_fp32_mb:.2f} MB")
print(f"Inference Latency       : {mean_latency * 1000:.2f} ms/image")
print(f"Throughput              : {1 / mean_latency:.2f} images/sec")
print(f"Energy Proxy            : {mean_latency * params:.2e}")
print("=" * 60)